# Chapter 6: Merging, Joining & Concatenation

Combining multiple DataFrames is essential for real-world analysis.

1. `pd.concat` — stacking DataFrames
2. `pd.merge` — SQL-style joins
3. Merge types: inner, left, right, outer, cross
4. Handling duplicate columns with `suffixes`
5. Validating merges
6. Practical example: customers, orders, products

In [ ]:
import pandas as pd
import numpy as np

## 6.1 `pd.concat` — Stacking DataFrames

In [ ]:
# Vertical stacking (axis=0) — most common
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]}, index=[0, 1])
df2 = pd.DataFrame({'A': [5, 6], 'B': [7, 8]}, index=[2, 3])

result = pd.concat([df1, df2])
result

In [ ]:
# ignore_index to reset the index
pd.concat([df1, df2], ignore_index=True)

In [ ]:
# Horizontal stacking (axis=1)
df3 = pd.DataFrame({'C': [10, 20]}, index=[0, 1])
pd.concat([df1, df3], axis=1)

In [ ]:
# Handling mismatched columns
df_a = pd.DataFrame({'X': [1, 2], 'Y': [3, 4]})
df_b = pd.DataFrame({'Y': [5, 6], 'Z': [7, 8]})

print('join="outer" (default):')
print(pd.concat([df_a, df_b], ignore_index=True))

print('\njoin="inner":')
print(pd.concat([df_a, df_b], ignore_index=True, join='inner'))

## 6.2 `pd.merge` — SQL-Style Joins

The `pd.merge()` function is equivalent to SQL `JOIN`.

### Sample Tables

In [ ]:
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'region': ['North', 'South', 'East', 'West', 'North'],
})

orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106],
    'customer_id': [1, 2, 2, 3, 5, 6],  # note: customer 6 has no match!
    'product_id': [1, 2, 1, 3, 2, 1],
    'amount': [100, 200, 150, 300, 250, 50],
})

products = pd.DataFrame({
    'product_id': [1, 2, 3],
    'product_name': ['Widget', 'Gadget', 'Doohickey'],
    'price': [9.99, 24.99, 4.99],
})

print('Customers:'); display(customers)
print('Orders:'); display(orders)
print('Products:'); display(products)

## 6.3 Join Types

| Type | SQL Equivalent | Result |
|------|---------------|--------|
| `inner` | INNER JOIN | Only matching rows |
| `left` | LEFT JOIN | All from left + matches from right |
| `right` | RIGHT JOIN | All from right + matches from left |
| `outer` | FULL OUTER JOIN | All rows from both |
| `cross` | CROSS JOIN | Cartesian product |

In [ ]:
# INNER JOIN — only matching customer_ids
inner = pd.merge(orders, customers, on='customer_id', how='inner')
print(f'Inner join: {len(inner)} rows (order 106 dropped — no matching customer)')
inner

In [ ]:
# LEFT JOIN — keep all orders
left = pd.merge(orders, customers, on='customer_id', how='left')
print(f'Left join: {len(left)} rows (order 106 has NaN for customer info)')
left

In [ ]:
# RIGHT JOIN — keep all customers
right = pd.merge(orders, customers, on='customer_id', how='right')
print(f'Right join: {len(right)} rows (Diana has NaN for order info)')
right

In [ ]:
# OUTER JOIN — keep everything
outer = pd.merge(orders, customers, on='customer_id', how='outer')
print(f'Outer join: {len(outer)} rows')
outer

### Merging on Different Column Names

In [ ]:
# When column names differ between tables
df_left = pd.DataFrame({'emp_id': [1, 2, 3], 'name': ['A', 'B', 'C']})
df_right = pd.DataFrame({'employee_id': [1, 2, 4], 'dept': ['HR', 'Eng', 'Sales']})

pd.merge(df_left, df_right, left_on='emp_id', right_on='employee_id', how='inner')

### Merging on Index

In [ ]:
df_idx = customers.set_index('customer_id')

# Merge using left index and right column
pd.merge(df_idx, orders, left_index=True, right_on='customer_id', how='inner').head()

## 6.4 Handling Duplicate Column Names

In [ ]:
# Both DataFrames have an 'amount' column
df_x = pd.DataFrame({'id': [1, 2], 'amount': [100, 200]})
df_y = pd.DataFrame({'id': [1, 2], 'amount': [10, 20]})

merged = pd.merge(df_x, df_y, on='id', suffixes=('_original', '_discount'))
merged

## 6.5 Validating Merges

In [ ]:
# Validate cardinality to catch unexpected duplicates
try:
    pd.merge(orders, customers, on='customer_id',
             how='inner', validate='one_to_one')
except Exception as e:
    print(f'Validation error: {e}')
    print('\nThis is expected — Bob has 2 orders (many-to-one)!')

In [ ]:
# Correct validation: many-to-one
result = pd.merge(orders, customers, on='customer_id',
                  how='inner', validate='many_to_one')
print(f'Validated merge: {len(result)} rows')

## 6.6 Practical Example: Full Join of Customers → Orders → Products

In [ ]:
# Step 1: Merge orders with customers
order_details = pd.merge(orders, customers, on='customer_id', how='left')

# Step 2: Merge with products
full_data = pd.merge(order_details, products, on='product_id', how='left')

# Step 3: Calculate total price
full_data['total_price'] = full_data['amount'] * full_data['price']

full_data[['order_id', 'name', 'product_name', 'amount', 'price', 'total_price', 'region']]

In [ ]:
# Analysis: Revenue by region
full_data.groupby('region')['total_price'].sum().sort_values(ascending=False)

## 📝 Key Takeaways

| Topic | Key Point |
|-------|----------|
| `pd.concat` | Stack DataFrames vertically (`axis=0`) or horizontally (`axis=1`) |
| `pd.merge` | SQL-style joins — `inner`, `left`, `right`, `outer` |
| `on` | Common column name in both tables |
| `left_on`/`right_on` | When column names differ |
| `suffixes` | Resolve duplicate column names |
| `validate` | Catch cardinality errors early |